In [1]:
import numpy as np
import math
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import t
from scipy import special
from scipy.interpolate import interp1d
from scipy.optimize import brentq, minimize,minimize_scalar

class t_Copula:
    def __init__(self, M, R, rho, T, delta, n_herm, v1, v2):
        self.M = M
        self.R = R
        self.rho = rho
        self.T = T
        self.delta = delta
        self.v1 = v1
        self.v2 = v2
        self.n_herm = n_herm

        self.nodes, self.weights = np.polynomial.hermite.hermgauss(n_herm)

        self.N = int(self.T / self.delta)

        self.times = np.arange(0, self.N + 1) * self.delta

        #Load rates
        df = pd.read_excel(r"C:\Speciale\rates_20-06-2025.xlsx")
        rates = []
        for i in range(1,len(df.columns)):
            rates.append(df.iloc[0,i])

        #Load index spreads
        df = pd.read_excel(r"C:\Speciale\index_spread_20-06-2025.xlsx")
        index_spread = (df.iloc[0, 1:])/10000
        index_spread = index_spread.tolist()
        self.index_spread = index_spread

        #Calculate default probabilities
        rate_times = np.arange(0, len(rates)) * 0.125
        self.zcb = np.exp(-np.array(rates)/100 * rate_times)

        default_probs = np.zeros(self.N+1)

        default_probs = np.zeros(self.N+1)
        maturities = [0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5]
        hazard_rates = self.bootstrap_hazard_rates(index_spread,maturities)
        h1 = hazard_rates[0]
        h2 = hazard_rates[1]
        h3 = hazard_rates[2]
        h4 = hazard_rates[3]
        h5 = hazard_rates[4]
        h6 = hazard_rates[5]
        h7 = hazard_rates[6]
        h8 = hazard_rates[7]
        h9 = hazard_rates[8]
        h10 = hazard_rates[9]

        for i in range(3):
            default_probs[i] = 1-np.exp(-h1*i*self.delta)
        for i in range(3,5):
            default_probs[i] = 1-(1-default_probs[2])*np.exp(-h2*(i-2)*self.delta)
        for i in range(5,7):
            default_probs[i] = 1-(1-default_probs[4])*np.exp(-h3*(i-4)*self.delta)
        for i in range(7,9):
            default_probs[i] = 1-(1-default_probs[6])*np.exp(-h4*(i-6)*self.delta)
        for i in range(9,11):
            default_probs[i] = 1-(1-default_probs[8])*np.exp(-h5*(i-8)*self.delta)
        for i in range(11,13):
            default_probs[i] = 1-(1-default_probs[10])*np.exp(-h6*(i-10)*self.delta)
        for i in range(13,15):
            default_probs[i] = 1-(1-default_probs[12])*np.exp(-h7*(i-12)*self.delta)
        for i in range(15,17):
            default_probs[i] = 1-(1-default_probs[14])*np.exp(-h8*(i-14)*self.delta)
        for i in range(17,19):
            default_probs[i] = 1-(1-default_probs[16])*np.exp(-h9*(i-16)*self.delta)
        for i in range(19,21):
            default_probs[i] = 1-(1-default_probs[18])*np.exp(-h10*(i-18)*self.delta)

        self.default_probs = default_probs
        print(default_probs)
        
        self.default_barriers = self.quantile_func(np.clip(default_probs, 1e-12,1 - 1e-12))

    def bootstrap_hazard_rates(self, spreads, maturities):
        hazard_rates = []

        def survival(t, hazard_rates, maturities):
            total = 0.0
            prev_T = 0.0

            for h, T in zip(hazard_rates, maturities):
                if t <= T:
                    total += h * (t - prev_T)
                    return np.exp(-total)
                else:
                    total += h * (T - prev_T)
                    prev_T = T

            return np.exp(-total)

        for k in range(len(spreads)):

            s_k = spreads[k]
            T_k = maturities[k]

            def cds_equation(hk):

                temp_hazards = hazard_rates + [hk]
                temp_maturities = maturities[:k+1]

                total = 0.0

                N = int(T_k / self.delta)

                for i in range(1, N + 1):
                    Ti = i * self.delta
                    Ti_m1 = (i - 1) * self.delta

                    # discount factor
                    P = self.zcb[2*i]

                    # survival probabilities
                    S_i = survival(Ti, temp_hazards, temp_maturities)
                    S_im1 = survival(Ti_m1, temp_hazards, temp_maturities)

                    # CDS equation 
                    total += P * (
                        (1 - self.R) * (S_im1 - S_i)
                        - s_k * self.delta * (S_im1 + S_i) / 2
                    )

                return total

            # solve for hk
            hk = brentq(cds_equation, 1e-6, 5.0)
            hazard_rates.append(hk)

        return hazard_rates

    def t_characteristic_function(self,v,t):
        t = np.atleast_1d(t)
        t_abs = np.abs(t)
        res = np.ones_like(t_abs)
        mask = t_abs > 1e-12
        arg = np.sqrt(v) * t_abs[mask]
        

        # The t-distribution CF formula
        log_bessel = np.log(special.kve(v/2,arg)) - arg

        log_numerator = (v/2) * np.log(arg) + log_bessel
        log_denominator = special.gammaln(v/2) + (v/2 - 1)* np.log(2)

        res[mask] = np.exp(log_numerator - log_denominator)
        return res

    def char_func_prod(self, t):
        # Standardizing factors
        scale1 = np.sqrt((self.v1 - 2) / self.v1) if self.v1 > 2 else 1.0
        scale2 = np.sqrt((self.v2 - 2) / self.v2) if self.v2 > 2 else 1.0
        
        # Apply scales to the characteristic functions
        phi1 = self.t_characteristic_function(self.v1, t * np.sqrt(self.rho) * scale1)
        phi2 = self.t_characteristic_function(self.v2, t * np.sqrt(1 - self.rho) * scale2)
        
        return phi1 * phi2

    def recover_all(self, N=2**16, L=30):
        dx = 2 * L / N
        x = np.linspace(-L, L, N, endpoint=False)
        
        dt = 2 * np.pi / (2 * L)
        t = np.fft.ifftshift(np.arange(-N/2, N/2)) * dt
        
        phi = self.char_func_prod(t)
        
        density = np.fft.ifft(phi) * (N * dt / (2 * np.pi))
        density = np.real(np.fft.ifftshift(density))
        
        mean = np.trapezoid(density * x, x)
        var = np.trapezoid(density * (x**2), x) - mean**2
        cdf = np.cumsum(density) * dx
        
        return x, density, cdf, mean, var
    
    def quantile_func(self,q):
        x, density, cdf, mean, var = self.recover_all() 
        inv_cdf = interp1d(cdf, x, bounds_error=False, fill_value="extrapolate")
        return inv_cdf(q)
    
    def cdf_function(self,x_input):
        x_grid, density, cdf_vals, mean, var = self.recover_all() 
        cdf_interp = interp1d(x_grid, cdf_vals, bounds_error=False, fill_value="extrapolate")
        return cdf_interp(x_input)
    
    
    def integrate_gauss_hermite(self,A,B,func):
        nodes = self.nodes
        weights = self.weights
        nu = self.v1
        
        mu = self.calc_mode(A,B,func)
        sigma_lp = self.calc_obs_inf(A,B,mu,func)
        if sigma_lp <= 1e-10:
            sigma_lp = 1.0  # Default scale
            mu = 0.0        # Default center

        t_nodes = mu + np.sqrt(2)* sigma_lp*nodes
        total = 0.0
        for i in range(self.n_herm):
            g = func(A,B,t_nodes[i]) * t.pdf(t_nodes[i],df = nu)
            gaussian_weight = weights[i] * np.exp(nodes[i]**2)
            total += g * gaussian_weight
        
        return np.sqrt(2) * sigma_lp * total
    
    def calc_mode(self, A, B, func):
        def f(x):
            val = func(A, B, x) * t.pdf(x, self.v1)
            return -val if val > 0 else 0.0
        best_mode = 0
        min_val = f(0)
        for start in [-2, 0, 2]:
            res = minimize(f, x0=start, bounds=[(-10, 10)])
            if res.fun < min_val:
                min_val = res.fun
                best_mode = res.x.item()
        return best_mode
        
    def calc_obs_inf(self, A,B, mode, func):
        def log_g(x):
            val = func(A, B, x)
            pdf = t.pdf(x, df=self.v1)

            val = max(val, 1e-300)
            pdf = max(pdf, 1e-300)

            return np.log(val) + np.log(pdf)

        def second_derivative(f, x, h=1e-4):
            return -(f(x + 2*h) + f(x - 2*h) - 2*f(x)) / (4*h**2)

        j = second_derivative(log_g, mode)

        if j > 0:
            return 1 / np.sqrt(j)
        else:
            def second_derivative(f, x, h=1e-3):
                return -(f(x + h) + f(x - h) - 2*f(x)) / (h**2)
            j = second_derivative(log_g, mode)
            if j > 0:
                return 1 / np.sqrt(j)
            else: 
                return 0.0
            

    
    def calc_expected_tranche_principal(self, A, B, S, k):
        S = np.asarray(S).item()
        S = float(S)
        barrier = self.default_barriers[k]

        # Standardizing factors
        scale1 = np.sqrt((self.v1 - 2) / self.v1) if self.v1 > 2 else 1.0
        scale2 = np.sqrt((self.v2 - 2) / self.v2) if self.v2 > 2 else 1.0

        sqrt_rho = np.sqrt(self.rho)
        sqrt_1mr = np.sqrt(1 - self.rho)

        numerator = barrier - (sqrt_rho * S * scale1)
        denominator = sqrt_1mr * scale2
        
        p = t.cdf(numerator / denominator, df=self.v2)
        p = min(max(p, 1e-16), 1 - 1e-16)

        M = self.M
        R = self.R

        nl = A * M / (1 - R)
        nh = B * M / (1 - R)

        il = min(math.ceil(nl), M+1)
        ih = min(math.ceil(nh), M+1)

        # compute binomial probabilities recursively
        probs = np.zeros(M+1)
        probs[0] = (1 - p)**M

        ratio = p/(1-p)

        for n in range(M):
            probs[n+1] = probs[n] * (M-n)/(n+1) * ratio

        sum1 = probs[:il].sum()

        sum2 = 0.0
        for j in range(il, ih):
            loss_fraction = (B - j*(1-R)/M)/(B-A)
            sum2 += probs[j] * loss_fraction

        return sum1 + sum2
    
    def plot_expected_tranche_principal(self,A,B):
        N = self.N
        def integrand(A,B,S):
            return self.calc_expected_tranche_principal(A, B, S, self.N)
        yvals = []
        xvals = []
        rho = np.linspace(0,1,100)
        old_rho = self.rho
        for i in rho:
            self.rho = i
            yvals.append(self.integrate_gauss_hermite(A,B,integrand))
            xvals.append(self.rho)

        plt.plot(xvals,yvals)
        plt.show()
        
    def calc_PV_regular_spread_payments(self,A,B,S):  #A in Hull
        delta = self.delta # assume equidistant payments
        N = self.N
        sum = 0
        for k in range(1, N+1):
            sum += delta * self.calc_expected_tranche_principal(A,B,S,k) * self.zcb[2*k]        
        return sum        
        
    def calc_accrual_payment(self,A,B,S): #B in Hull
        delta = self.delta # assume equidistant payments
        N = self.N
        sum = 0
        for k in range(1, N+1):
            sum += 0.5*delta*(self.calc_expected_tranche_principal(A,B,S,(k-1))-self.calc_expected_tranche_principal(A,B,S,k))*self.zcb[2*k-1]
        return sum
        
    
    def calc_PV_expected_payoffs(self,A,B,S):  #C in Hull
        N = self.N
        sum = 0
        for k in range(1, N+1):
            sum += (self.calc_expected_tranche_principal(A,B,S,k-1)-self.calc_expected_tranche_principal(A,B,S,k))*self.zcb[2*k-1]
            
        return sum

    
    def calc_fair_spread_and_upfront(self,A,B):
        fair_spread = float((self.integrate_gauss_hermite(A,B,self.calc_PV_expected_payoffs))/(self.integrate_gauss_hermite(A,B,self.calc_PV_regular_spread_payments)+self.integrate_gauss_hermite(A,B,self.calc_accrual_payment)))
        if fair_spread > 0.01:
            spread = 0.01
            upfront = float(self.integrate_gauss_hermite(A,B,self.calc_PV_expected_payoffs)-spread*(self.integrate_gauss_hermite(A,B,self.calc_PV_regular_spread_payments) + self.integrate_gauss_hermite(A,B,self.calc_accrual_payment)))
            return spread,upfront
        else:
            return fair_spread,0
        
    def tranche_spread_for_rho(self, A, B, rho):

        old_rho = self.rho
        self.rho = rho

        C = self.integrate_gauss_hermite(A,B,self.calc_PV_expected_payoffs)
        P = (
            self.integrate_gauss_hermite(A,B,self.calc_PV_regular_spread_payments)
            + self.integrate_gauss_hermite(A,B,self.calc_accrual_payment)
        )

        self.rho = old_rho

        return C / P
    
    def calc_compound_correlation(self, A, B, s, U):
        def objective(rho):
            def f(rho):
                old_rho = self.rho
                self.rho = rho

                C = self.integrate_gauss_hermite(A,B,self.calc_PV_expected_payoffs)
                P = (
                    self.integrate_gauss_hermite(A,B,self.calc_PV_regular_spread_payments)
                    + self.integrate_gauss_hermite(A,B,self.calc_accrual_payment)
                )

                self.rho = old_rho

                return U + s * P - C
            return (f(rho))**2

        res = minimize_scalar(objective, bounds=(0.0001,0.9999), method='bounded')
        return res.x
    
    def calc_base_correlation(self, target_aq, market_quotes):
        total_loss_0_aq = 0.0
        
        for (A, B, s, U) in market_quotes:
            if B > target_aq:
                break
                
            comp_rho = self.calc_compound_correlation(A, B, s, U)
            
            old_rho = self.rho
            self.rho = comp_rho
            
            C_q_absolute = self.integrate_gauss_hermite(A,B,self.calc_PV_expected_payoffs) * (B - A)
            total_loss_0_aq += C_q_absolute
            
            self.rho = old_rho
            
        def objective(rho):
            old_rho = self.rho
            self.rho = rho
            model_loss = self.integrate_gauss_hermite(0,target_aq,self.calc_PV_expected_payoffs) * target_aq
            self.rho = old_rho
            return model_loss - total_loss_0_aq

        return brentq(objective, 0.0001, 0.9999)
    

sim = t_Copula(
    M = 125,
    R = 0.4,
    rho = 0.5,
    T = 5,
    delta = 1/4,
    n_herm=200,
    v1=4,
    v2=4)

market_info = [
    (0.00, 0.03, 0.01, 0.26346),
    (0.03, 0.06, 0.01, 0.03927),
    (0.06, 0.12, 0.009806, 0.0),
    (0.12, 1.00, 0.002678, 0.0)
]
sim.calc_base_correlation(1,market_info)

[0.         0.00015787 0.00031571 0.00058798 0.00086017 0.00182118
 0.00278127 0.00428433 0.00578513 0.00791147 0.01003327 0.01289495
 0.01574836 0.01923282 0.02270495 0.02661489 0.03050919 0.03352263
 0.03652671 0.04158158 0.04660994]


0.23657570974179126